In [12]:
import pandas as pd
import re
import time

df = pd.read_parquet("../output/sample.parquet")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()



551     Great improvement over the original yellow Gor...
721     This works great!  the adhesive holds strong a...
702     Not quick drying at all. I used it to repair t...
645     I love these for various personal areas (ahem…...
1470    Cant live without it next to me with my allerg...
Name: reviewText, dtype: string

In [ ]:
# Sample of raw review text (before cleaning)
df["reviewText"].sample(5, random_state=42)

In [ ]:
def clean_review_text_pandas(series: pd.Series) -> pd.Series:
    """Clean review text: lowercase, remove punctuation, normalize whitespace."""
    # Handle missing/NaN
    s = series.astype(str).fillna("")
    # Lowercase
    s = s.str.lower()
    # Keep only letters, numbers, spaces; replace other chars with space
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    # Collapse multiple spaces/newlines into one space
    s = s.str.replace(r"\s+", " ", regex=True)
    # Strip leading/trailing spaces
    s = s.str.strip()
    return s
#gpt
# Apply and time (CPU)
t0 = time.perf_counter()
df["reviewText_clean"] = clean_review_text_pandas(df["reviewText"])
cpu_time = time.perf_counter() - t0
print(f"Pandas (CPU) cleaning: {cpu_time:.3f} s for {len(df):,} rows")

In [ ]:
# Before vs after (sample)
sample_idx = df["reviewText"].dropna().index[:3].tolist()
for i in sample_idx:
    print("Before:", df.loc[i, "reviewText"][:80], "...")
    print("After: ", df.loc[i, "reviewText_clean"][:80], "...")
    print()

In [ ]:
HAS_CUDF = False
try:
    import cudf
    HAS_CUDF = True
except ImportError:
    pass

if HAS_CUDF:
    def clean_review_text_cudf(series):  # cudf.Series
        s = series.astype("str").fillna("")
        s = s.str.lower()
        s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
        s = s.str.replace(r"\s+", " ", regex=True)
        s = s.str.strip()
        return s

    gdf = cudf.from_pandas(df[["reviewText"]].copy())
    t0 = time.perf_counter()
    gdf["reviewText_clean_gpu"] = clean_review_text_cudf(gdf["reviewText"])
    gpu_time = time.perf_counter() - t0
    print(f"cuDF (GPU) cleaning: {gpu_time:.3f} s for {len(gdf):,} rows")
    if cpu_time > 0:
        print(f"Speedup vs Pandas: {cpu_time / gpu_time:.2f}x")
else:
    print("cuDF not available (no NVIDIA GPU or cudf not installed).")
    print("GPU step can be run when CUDA/cuDF environment is available.")
    print("Pandas (CPU) cleaned column 'reviewText_clean' is the deliverable for this week.")

In [ ]:
out_path = "../output/sample_clean.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved {len(df):,} rows to {out_path}")
print("Columns:", df.columns.tolist())